# Bronze Layer — Ingesta de Datos StackOverflow

**Proyecto 3 — Arquitectura Lakehouse en Azure Databricks**

## Flujo de ingesta (100% distribuido)
`spark.read.parquet(s3_url)` → filtra 2 meses en Spark → escribe Parquet en **Unity Catalog Volume** con override

## Solución al error `Invalid configuration value for fs.azure.account.key`
El cluster usa **Unity Catalog + Access Connector (OAuth/Managed Identity)** — no account keys.
Escribir directo a `abfss://` requiere configurar la account key en las Spark conf del cluster,
lo que entra en conflicto con Unity Catalog.

**Solución:** escribir a través de un **Volumen de Unity Catalog** (`/Volumes/catalog/schema/volume/`).
UC resuelve las credenciales automáticamente vía el Access Connector — cero configuración adicional.

**Fuente S3 (sin autenticación):**
```
https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/posts/2023.parquet
https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/votes/2023.parquet
https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/comments/2023.parquet
https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/users.parquet
https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/badges.parquet
https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/postlinks.parquet
```

**Tablas ingestadas:** posts, users, votes, comments, badges, postlinks

In [0]:
# ============================================================
# CELDA 1 — Parámetros de configuración
# ============================================================

# Catálogo Unity Catalog
CATALOG       = "lacm_uao_prod_central_us"
BRONZE_SCHEMA = "bronze"

# ---------------------------------------------------------------
# SOLUCIÓN AL ERROR: Invalid configuration value for fs.azure.account.key
#
# El error ocurría porque el código escribía directo con abfss://
# usando account key, pero el cluster usa Unity Catalog + Access
# Connector (OAuth / Managed Identity) — que NO usa account key.
#
# Solución: escribir a través de un Volumen de Unity Catalog.
# Los volúmenes de UC heredan automáticamente las credenciales
# del Access Connector sin necesidad de configurar account keys.
#
# Ruta de volumen UC: /Volumes/<catalog>/<schema>/<volume>/<path>
# Se monta automáticamente en el cluster, no requiere credenciales.
# ---------------------------------------------------------------

# Nombre del volumen a crear en Unity Catalog (bajo el esquema bronze)
BRONZE_VOLUME = "raw_data"

# Ruta de escritura via Unity Catalog Volume — NO requiere account key
BRONZE_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{BRONZE_VOLUME}"

# Ruta ADLS solo como referencia para la external location (NO se usa para escribir)
STORAGE_ACCOUNT = "stuaoprod001lacm"
CONTAINER       = "bronze"
ADLS_BASE       = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Período de ingesta — 2 meses del mismo año
YEAR   = 2023
MONTHS = [1, 2]   # Enero y Febrero 2023

# Fuente: S3 público de ClickHouse — SIN autenticación
S3_BASE = "https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet"

TABLE_DEFS = {
    "posts":     (f"{S3_BASE}/posts/{YEAR}.parquet",    "CreationDate"),
    "votes":     (f"{S3_BASE}/votes/{YEAR}.parquet",    "CreationDate"),
    "comments":  (f"{S3_BASE}/comments/{YEAR}.parquet", "CreationDate"),
    "users":     (f"{S3_BASE}/users.parquet",           "CreationDate"),
    "badges":    (f"{S3_BASE}/badges.parquet",          "Date"),
    "postlinks": (f"{S3_BASE}/postlinks.parquet",       "CreationDate"),
}

TABLES = list(TABLE_DEFS.keys())

print(f"[CONFIG] Catálogo:    {CATALOG}.{BRONZE_SCHEMA}")
print(f"[CONFIG] Volumen UC:  {BRONZE_PATH}")
print(f"[CONFIG] ADLS base:   {ADLS_BASE}")
print(f"[CONFIG] Período:     {MONTHS[0]:02d}/{YEAR} — {MONTHS[-1]:02d}/{YEAR}")
print(f"[CONFIG] Tablas:      {TABLES}")


[CONFIG] Catálogo:    lacm_uao_prod_central_us.bronze
[CONFIG] Volumen UC:  /Volumes/lacm_uao_prod_central_us/bronze/raw_data
[CONFIG] ADLS base:   abfss://bronze@stuaoprod001lacm.dfs.core.windows.net
[CONFIG] Período:     01/2023 — 02/2023
[CONFIG] Tablas:      ['posts', 'votes', 'comments', 'users', 'badges', 'postlinks']


In [0]:
# ============================================================
# CELDA 2 — Setup: imports, funciones, esquema y volumen UC
#
# ESTRATEGIA FINAL PARA SERVERLESS:
#   Serverless solo permite acceso a rutas /Volumes/ y /Workspace/
#   /tmp, /local_disk0 y file:/// están bloqueados por seguridad UC.
#
#   Flujo:
#   requests descarga el Parquet
#   → se guarda en /Volumes/.../bronze/tmp/ (permitido en Serverless)
#   → spark.read.parquet('/Volumes/.../bronze/tmp/archivo.parquet')
#   → filtra meses y escribe en /Volumes/.../bronze/raw_data/
# ============================================================
import time, os, requests
from collections import defaultdict
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    lit, current_timestamp, to_timestamp,
    year as sp_year, month as sp_month
)

spark = SparkSession.builder.getOrCreate()

# Directorio temporal DENTRO del volumen UC (único path permitido en Serverless)
TMP_DIR = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{BRONZE_VOLUME}/tmp"
os.makedirs(TMP_DIR, exist_ok=True)
print(f"[OK] Directorio temporal: {TMP_DIR}")


# ── Descarga S3 → /Volumes/tmp ────────────────────────────────────────────────
def download_to_volume(url: str, filename: str) -> str:
    """
    Descarga Parquet desde S3 público y lo guarda en /Volumes/.../tmp/
    Retorna la ruta /Volumes/ que Spark puede leer en Serverless.
    """
    vol_path = f"{TMP_DIR}/{filename}"

    if os.path.exists(vol_path):
        size_mb = os.path.getsize(vol_path) / 1048576
        print(f"  Reutilizando {filename} ({size_mb:.1f} MB ya en volumen)")
        return vol_path

    print(f"  Descargando {filename} ...", end=" ", flush=True)
    t0 = time.time()
    resp = requests.get(url, stream=True, timeout=600)
    resp.raise_for_status()
    mb = 0
    with open(vol_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=16 * 1024 * 1024):
            f.write(chunk)
            mb += len(chunk) / 1048576
    print(f"{mb:.1f} MB en {time.time()-t0:.1f}s")
    return vol_path


# ── Ingesta: descarga → /Volumes/tmp → Spark filtra → /Volumes/raw_data ───────
def ingest_table_spark(
    table_name: str,
    s3_url: str,
    date_col: str,
    year: int,
    months: list
) -> dict:
    """
    1. Descarga el Parquet a /Volumes/.../bronze/tmp/ con requests
    2. spark.read.parquet('/Volumes/...') — ruta permitida en Serverless
    3. Filtra por mes y escribe en /Volumes/.../bronze/raw_data/ con override
    4. Borra el archivo temporal del volumen
    """
    filename = f"{table_name}_{s3_url.split('/')[-1]}"

    # Paso 1: descargar al volumen UC
    vol_path = download_to_volume(s3_url, filename)

    # Paso 2: leer con Spark desde /Volumes/ (permitido en Serverless)
    t0 = time.time()
    df_full = spark.read.parquet(vol_path)
    df_full = df_full.withColumn(date_col, to_timestamp(date_col))
    print(f"  Schema OK en {time.time()-t0:.1f}s")

    results = {}
    for m in months:
        t1 = time.time()
        df_month = (
            df_full
            .filter((sp_year(date_col) == year) & (sp_month(date_col) == m))
            .withColumn("ingest_year",  lit(year))
            .withColumn("ingest_month", lit(m))
            .withColumn("ingest_ts",    current_timestamp())
        )
        out_path = f"{BRONZE_PATH}/{table_name}/year={year}/month={m:02d}"
        df_month.write.mode("overwrite").parquet(out_path)
        n = spark.read.parquet(out_path).count()
        elapsed_w = time.time() - t1
        if n == 0:
            print(f"  [WARN] {table_name} {year}-{m:02d}: 0 filas")
        else:
            print(f"  [OK] {table_name} {year}-{m:02d}: {n:,} filas → {out_path} ({elapsed_w:.1f}s)")
        results[m] = n

    # Paso 4: limpiar el temporal del volumen para liberar espacio
    try:
        os.remove(vol_path)
        print(f"  [CLEAN] {filename} eliminado de /tmp volumen")
    except Exception as e:
        print(f"  [WARN] No se pudo limpiar {filename}: {e}")

    return results


# ── Preparar esquema y volumen en Unity Catalog ───────────────────────────────
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
print(f"[OK] Esquema listo: {CATALOG}.{BRONZE_SCHEMA}")

try:
    spark.sql(f"""
        CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_VOLUME}
        LOCATION '{ADLS_BASE}'
    """)
    print(f"[OK] Volumen externo listo: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_VOLUME}")
except Exception as e:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_VOLUME}")
    print(f"[OK] Volumen managed listo: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_VOLUME}")

print(f"[OK] Ruta Bronze: {BRONZE_PATH}")
print("[INIT] Setup completo — ingest_table_spark lista para usar.")


[OK] Esquema listo: lacm_uao_prod_central_us.bronze
[OK] Volumen externo listo: lacm_uao_prod_central_us.bronze.raw_data
[OK] Ruta de escritura: /Volumes/lacm_uao_prod_central_us/bronze/raw_data
[OK] Directorio temporal: /local_disk0/tmp/stackoverflow
[INIT] Setup completo — ingest_table_spark lista para usar.


In [0]:
# ============================================================
# CELDA 3 — Ingesta principal: S3 → Spark distribuido → UC Volume
#
# Requiere haber ejecutado primero la CELDA 2 (setup).
# ============================================================
ingest_summary = defaultdict(dict)
errors = []

print(f"[START] Ingesta Bronze — {YEAR} meses {MONTHS}")
print(f"[START] Destino: {BRONZE_PATH}")
print("=" * 65)

for table_name, (url, date_col) in TABLE_DEFS.items():
    print(f"\n[TABLE] {table_name.upper()}")
    try:
        counts = ingest_table_spark(
            table_name=table_name,
            s3_url=url,
            date_col=date_col,
            year=YEAR,
            months=MONTHS
        )
        ingest_summary[table_name] = counts

        # Registrar tabla Delta en Unity Catalog usando saveAsTable
        # (sin LOCATION — /Volumes/ no es un cloud file system scheme)
        table_full = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
        vol_path   = f"{BRONZE_PATH}/{table_name}"
        spark.read.parquet(vol_path) \
            .write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_full)
        print(f"  [UC] Tabla Delta registrada: {table_full}")

    except Exception as e:
        msg = f"[ERROR] {table_name}: {e}"
        print(msg)
        errors.append(msg)
        ingest_summary[table_name] = {m: -1 for m in MONTHS}

print("\n" + "=" * 65)
print("[DONE] Ingesta Bronze finalizada.")


[START] Ingesta Bronze — 2023 meses [1, 2]
[START] Destino: /Volumes/lacm_uao_prod_central_us/bronze/raw_data

[TABLE] POSTS
  Descargando posts_2023.parquet ... 1567.2 MB en 64.1s
  Schema OK en 0.2s (['Id', 'PostTypeId', 'AcceptedAnswerId', 'CreationDate']...)
  [OK] posts 2023-01: 225,487 filas → /Volumes/lacm_uao_prod_central_us/bronze/raw_data/posts/year=2023/month=01 (13.8s)
  [OK] posts 2023-02: 198,825 filas → /Volumes/lacm_uao_prod_central_us/bronze/raw_data/posts/year=2023/month=02 (10.6s)
  [UC] Tabla Delta registrada: lacm_uao_prod_central_us.bronze.posts

[TABLE] VOTES
  Descargando votes_2023.parquet ... 98.9 MB en 5.1s
  Schema OK en 0.1s (['Id', 'PostId', 'VoteTypeId', 'CreationDate']...)
  [OK] votes 2023-01: 1,256,117 filas → /Volumes/lacm_uao_prod_central_us/bronze/raw_data/votes/year=2023/month=01 (10.2s)
  [OK] votes 2023-02: 1,129,914 filas → /Volumes/lacm_uao_prod_central_us/bronze/raw_data/votes/year=2023/month=02 (7.7s)
  [UC] Tabla Delta registrada: lacm_uao_p

In [0]:
# ============================================================
# CELDA 4 — Reporte de resultados y validación
# ============================================================
print("\n" + "=" * 60)
print("RESUMEN DE INGESTA BRONZE")
print("=" * 60)
print(f"{'Tabla':<13} | {f'{MONTHS[0]:02d}/{YEAR}':>10} | {f'{MONTHS[1]:02d}/{YEAR}':>10} | {'Total':>10}")
print("-" * 52)

total_rows = 0
for table in TABLES:
    m1 = ingest_summary[table].get(MONTHS[0], 0)
    m2 = ingest_summary[table].get(MONTHS[1], 0)
    subtotal = (m1 if m1 > 0 else 0) + (m2 if m2 > 0 else 0)
    total_rows += subtotal
    m1_str = f"{m1:,}" if m1 >= 0 else "ERROR"
    m2_str = f"{m2:,}" if m2 >= 0 else "ERROR"
    print(f"{table:<13} | {m1_str:>10} | {m2_str:>10} | {subtotal:>10,}")

print("-" * 52)
print(f"{'TOTAL':<13} | {'':>10} | {'':>10} | {total_rows:>10,}")

if errors:
    print(f"\n[WARN] {len(errors)} error(es):")
    for err in errors:
        print(f"  {err}")
else:
    print("\n[OK] Ingesta completada sin errores.")

print(f"\n[INFO] Ruta Bronze: {BRONZE_PATH}")
print(f"[INFO] Catálogo:    {CATALOG}.{BRONZE_SCHEMA}")

# ── Validación física ────────────────────────────────────────
print("\n" + "=" * 60)
print("VALIDACIÓN — Verificando datos en volumen Bronze")
print("=" * 60)

for table in TABLES:
    try:
        path = f"{BRONZE_PATH}/{table}/year={YEAR}/month={MONTHS[0]:02d}"
        df_check = spark.read.parquet(path)
        print(f"✓ {table}: {df_check.count():,} filas — {len(df_check.columns)} columnas")
    except Exception as e:
        print(f"✗ {table}: {e}")

print("\n[SAMPLE] Primeras 5 filas de posts (mes 1):")
spark.read.parquet(f"{BRONZE_PATH}/posts/year={YEAR}/month={MONTHS[0]:02d}") \
     .select("Id", "PostTypeId", "CreationDate", "Score", "Title") \
     .limit(5).show(truncate=50)



RESUMEN DE INGESTA BRONZE
Tabla         |    01/2023 |    02/2023 |      Total
----------------------------------------------------
posts         |    225,487 |    198,825 |    424,312
votes         |  1,256,117 |  1,129,914 |  2,386,031
comments      |    346,271 |    300,332 |    646,603
users         |    216,342 |    183,700 |    400,042
badges        |    336,767 |    290,359 |    627,126
postlinks     |     32,410 |     27,592 |     60,002
----------------------------------------------------
TOTAL         |            |            |  4,544,116

[OK] Ingesta completada sin errores.

[INFO] Ruta Bronze: /Volumes/lacm_uao_prod_central_us/bronze/raw_data
[INFO] Catálogo:    lacm_uao_prod_central_us.bronze

VALIDACIÓN — Verificando datos en volumen Bronze
✓ posts: 225,487 filas — 25 columnas
✓ votes: 1,256,117 filas — 9 columnas
✓ comments: 346,271 filas — 10 columnas
✓ users: 216,342 filas — 15 columnas
✓ badges: 336,767 filas — 9 columnas
✓ postlinks: 32,410 filas — 8 columnas

[SA